# Phase 3: Fine-tuned RoBERTa (Google Colab Version)

**Run this on Google Colab with a T4 GPU.**

## Setup steps:
1. Open https://colab.research.google.com/ → File → Upload notebook → upload this file
2. Runtime → Change runtime type → **T4 GPU** (free)
3. Upload `train.csv` to the Colab file browser (left sidebar → upload icon)
4. Run all cells (Runtime → Run all)
5. Total time: ~15-25 minutes

## What this does
Fine-tunes pre-trained RoBERTa-base on the Jigsaw toxic comment dataset for multilabel classification.

## Verify GPU is available

In [ ]:
!nvidia-smi

## Install dependencies

Most are pre-installed on Colab. Just need `transformers`.

In [ ]:
!pip install -q transformers

## Imports

In [ ]:
import re
import json
import time
from collections import Counter
from pathlib import Path
from typing import Tuple, Dict, List, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score

from transformers import RobertaModel, RobertaTokenizer, get_linear_schedule_with_warmup

import warnings
warnings.filterwarnings('ignore')

# Verify CUDA
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

LABEL_COLS = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

## Load and split data

Make sure you uploaded `train.csv` to Colab. Click the folder icon on the left sidebar to verify it's there.

In [ ]:
# Load data
df = pd.read_csv('train.csv')
df['comment_text'] = df['comment_text'].fillna('')
print(f'Loaded {len(df):,} comments')

# Stratified split (same as Mac version)
stratify_key = df[LABEL_COLS].astype(str).agg(''.join, axis=1)
combo_counts = stratify_key.value_counts()
rare = combo_counts[combo_counts < 2].index
stratify_key = stratify_key.replace(rare, 'rare')

train_df, temp_df, strat_train, strat_temp = train_test_split(
    df, stratify_key, test_size=0.2, random_state=42, stratify=stratify_key
)

strat_temp = strat_temp.reset_index(drop=True)
temp_df = temp_df.reset_index(drop=True)
temp_combo_counts = strat_temp.value_counts()
rare_temp = temp_combo_counts[temp_combo_counts < 2].index
strat_temp = strat_temp.replace(rare_temp, 'rare')

val_df, test_df = train_test_split(
    temp_df, test_size=0.5, random_state=42, stratify=strat_temp
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f'Train: {len(train_df):,}  Val: {len(val_df):,}  Test: {len(test_df):,}')

## Model definition

In [ ]:
class RoBERTaClassifier(nn.Module):
    def __init__(self, num_labels=6, model_name='roberta-base', dropout=0.1):
        super().__init__()
        self.roberta = RobertaModel.from_pretrained(model_name)
        hidden_size = self.roberta.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size, num_labels)

    def forward(self, input_ids, attention_mask):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]
        return self.classifier(self.dropout(cls_output))


class FocalLoss(nn.Module):
    def __init__(self, alpha=1.0, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        bce_loss = nn.functional.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        probs = torch.sigmoid(logits)
        p_t = probs * targets + (1 - probs) * (1 - targets)
        focal_weight = (1 - p_t) ** self.gamma
        return (self.alpha * focal_weight * bce_loss).mean()


class ToxicDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=256):
        self.texts = df['comment_text'].tolist()
        self.labels = df[LABEL_COLS].values.astype(np.float32)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels': torch.tensor(self.labels[idx], dtype=torch.float32),
        }

## Build datasets and DataLoaders

Larger batch size now (16) since T4 has 16 GB VRAM.

In [ ]:
tokenizer = RobertaTokenizer.from_pretrained('roberta-base')

MAX_LENGTH = 256
BATCH_SIZE = 16  # T4 GPU can handle this easily

train_dataset = ToxicDataset(train_df, tokenizer, max_length=MAX_LENGTH)
val_dataset = ToxicDataset(val_df, tokenizer, max_length=MAX_LENGTH)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f'Train batches: {len(train_loader):,}  Val batches: {len(val_loader):,}')

## Create model, optimizer, scheduler

In [ ]:
model = RoBERTaClassifier(num_labels=6).to(device)
loss_fn = FocalLoss(gamma=2.0)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)

n_epochs = 3
total_steps = len(train_loader) * n_epochs
warmup_steps = int(0.1 * total_steps)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

total_params = sum(p.numel() for p in model.parameters())
print(f'Model loaded. Total params: {total_params:,}')
print(f'Training: {n_epochs} epochs, {total_steps:,} total steps, {warmup_steps:,} warmup steps')

## Training loop with progress bar

In [ ]:
from tqdm.auto import tqdm

history = {'train_loss': [], 'val_loss': []}
best_val_loss = float('inf')

for epoch in range(n_epochs):
    # Training
    model.train()
    total_loss = 0
    n_batches = 0
    
    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{n_epochs} [train]')
    for batch in pbar:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        logits = model(input_ids, attention_mask)
        loss = loss_fn(logits, labels)
        
        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        
        total_loss += loss.item()
        n_batches += 1
        pbar.set_postfix(loss=f'{loss.item():.4f}')
    
    train_loss = total_loss / n_batches
    history['train_loss'].append(train_loss)
    
    # Validation
    model.eval()
    val_loss = 0
    n_batches = 0
    
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f'Epoch {epoch+1}/{n_epochs} [val]'):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            logits = model(input_ids, attention_mask)
            loss = loss_fn(logits, labels)
            val_loss += loss.item()
            n_batches += 1
    
    val_loss /= n_batches
    history['val_loss'].append(val_loss)
    
    marker = ' (new best)' if val_loss < best_val_loss else ''
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'roberta_best.pt')
    
    print(f'Epoch {epoch+1}: train_loss={train_loss:.4f}  val_loss={val_loss:.4f}{marker}')

# Restore best model
model.load_state_dict(torch.load('roberta_best.pt'))
print(f'\nDone. Best val loss: {best_val_loss:.4f}')

## Evaluate with threshold tuning

In [ ]:
model.eval()
all_probs = []
all_labels = []

with torch.no_grad():
    for batch in tqdm(val_loader, desc='Evaluating'):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        logits = model(input_ids, attention_mask)
        all_probs.append(torch.sigmoid(logits).cpu().numpy())
        all_labels.append(labels.cpu().numpy())

all_probs = np.concatenate(all_probs, axis=0)
all_labels = np.concatenate(all_labels, axis=0)

# Try different thresholds
print('Threshold tuning:')
best_thresh = 0.5
best_f1 = 0
for thresh in [0.30, 0.35, 0.40, 0.45, 0.50]:
    preds = (all_probs >= thresh).astype(int)
    f1s = [f1_score(all_labels[:, i], preds[:, i], zero_division=0) for i in range(6)]
    macro_f1 = np.mean(f1s)
    marker = ' <- best' if macro_f1 > best_f1 else ''
    if macro_f1 > best_f1:
        best_f1 = macro_f1
        best_thresh = thresh
    print(f'  threshold={thresh:.2f}  macro_f1={macro_f1:.4f}{marker}')

# Final results with best threshold
preds = (all_probs >= best_thresh).astype(int)
print(f'\n=== Final results (threshold={best_thresh}) ===')
print(f'{"Label":<15} {"F1":>6} {"Prec":>6} {"Rec":>6} {"AUC":>6}')
results = {'per_label': {}, 'macro': {}}
f1s, precs, recs, aucs = [], [], [], []
for i, label in enumerate(LABEL_COLS):
    f1 = f1_score(all_labels[:, i], preds[:, i], zero_division=0)
    prec = precision_score(all_labels[:, i], preds[:, i], zero_division=0)
    rec = recall_score(all_labels[:, i], preds[:, i], zero_division=0)
    auc = roc_auc_score(all_labels[:, i], all_probs[:, i])
    f1s.append(f1); precs.append(prec); recs.append(rec); aucs.append(auc)
    results['per_label'][label] = {'f1': round(f1,4), 'precision': round(prec,4), 'recall': round(rec,4), 'roc_auc': round(auc,4), 'support': int(all_labels[:,i].sum())}
    print(f'{label:<15} {f1:>6.4f} {prec:>6.4f} {rec:>6.4f} {auc:>6.4f}')

results['macro'] = {'f1': round(np.mean(f1s),4), 'precision': round(np.mean(precs),4), 'recall': round(np.mean(recs),4), 'roc_auc': round(np.mean(aucs),4)}
results['model'] = 'roberta'
results['best_threshold'] = best_thresh
print(f'\n{"MACRO":<15} {np.mean(f1s):>6.4f} {np.mean(precs):>6.4f} {np.mean(recs):>6.4f} {np.mean(aucs):>6.4f}')

# Save results
with open('roberta_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print('\nResults saved to roberta_results.json')

## Compare with previous models

(Hardcoded for comparison)

In [ ]:
print('=== MODEL COMPARISON ===')
print(f'{"Model":<25}  {"Macro F1":>9} {"Macro AUC":>10}')
print('-' * 50)
print(f'{"TF-IDF + LogReg":<25}  {0.5633:>9.4f} {0.9810:>10.4f}')
print(f'{"BiLSTM + GloVe":<25}  {0.5947:>9.4f} {0.9730:>10.4f}')
print(f'{"RoBERTa fine-tuned":<25}  {results["macro"]["f1"]:>9.4f} {results["macro"]["roc_auc"]:>10.4f}')
print()
improvement = (results["macro"]["f1"] - 0.5633) / 0.5633 * 100
print(f'RoBERTa vs baseline: +{improvement:.1f}% relative improvement')

## Download results

Download `roberta_results.json` and `roberta_best.pt` to your computer:

In [ ]:
from google.colab import files
files.download('roberta_results.json')
# files.download('roberta_best.pt')  # ~500 MB — uncomment if you want the model